In [1]:
# split instance name

def igroup(x):
    if x[0:3] == 'cls':
        res = x.split('0')
        return res[0]
    return x
    

def inode(x):
    if x[0:3] == 'cls':
        res = x.split('0')
        return res[1].split(':')[0]
    return '1'
    

In [34]:
def get_instances(pf):
    instances = []
    for index,row in pf.iterrows():
        try:
            instances.append(json.loads(row['metrics'])['metric']['instance'])
        except Exception as e:
            #print(json.loads(row['metrics']))
            pass
    return set(instances)

In [72]:
import pandas as pd
def get_df_single_metrics(pf,prometheus_file,instances_select):
    df_single_metrics =  pd.DataFrame(columns=['name','igroup','inode','metric','values','timestamp','id'])
    prometheus_file_split = prometheus_file.split('/')[-1].split('_')[-2:]
    timestamp = prometheus_file_split[0]
    fid = prometheus_file_split[1].split('.')[0]

    for index,row in pf.iterrows():
        try:
            data = json.loads(row['metrics'])
            if data['metric']['instance'] in instances_select:
                df_single_metrics.loc[index] = data['metric']['__name__'], igroup(data['metric']['instance']),inode(data['metric']['instance']), data['metric'],data['values'],timestamp, fid
        except Exception as e:
            #print(e)
            df_single_metrics.loc[index] = data['metric']['__name__'], '', '', data['metric'],data['values'],timestamp, fid
    return df_single_metrics

In [24]:
prometheus_files = list(spark.read.options(delimiter=',') \
                      .csv('hdfs://clspromon-aio01.txx.seeburger.de/prometheus_files.txt').toPandas().iloc[:, 0])

In [75]:
prometheus_file = prometheus_files[-1]
prometheus_file

'hdfs://172.30.17.145:8020/prometheus_data/see_cloud_prometheus_single_metrics_1602752343_b18b1c2f-23fa-4e05-b192-184d36968dcb.seq'

In [2]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate();
metrics = sc.sequenceFile(prometheus_file).map(lambda x: x[1])

In [5]:
from pyspark.sql import SparkSession
spark = SparkSession(sc)

In [29]:
from pyspark.sql.types import Row
row = Row("metrics")
df = metrics.map(row).toDF()

In [30]:
pf = df.toPandas()

In [77]:
from notebook.auth import passwd
my_password = "GnP4<+T5yKeu"
hashed_password = passwd(passphrase=my_password, algorithm='sha256')
print(hashed_password)

sha256:c5a96f0759e9:7abb877a8c78504ac1d37cf8e0e80afbb88d1cbb75bc52c37a0dcacdcfcc4889


In [35]:
instances = get_instances(pf)

In [40]:
instances_select = ['clspromsg-cfs01:13000','clspromsg-cfs02:13000','clspromsg-cfs03:13000',
'clspromsg-edi01:13000','clspromsg-edi02:13000']

In [73]:
df_single_metrics = get_df_single_metrics(pf,prometheus_file,instances_select)

In [74]:
df_single_metrics

,name,igroup,inode,metric,values,timestamp,id
0,ALERTS,,,"{'__name__': 'ALERTS', 'alertname': 'ServiceCo...","[[1602752433, 1], [1602752463, 1], [1602752493...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
1,ALERTS,,,"{'__name__': 'ALERTS', 'alertname': 'ServiceCo...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
2,ALERTS,,,"{'__name__': 'ALERTS', 'alertname': 'ServiceCo...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
3,ALERTS,,,"{'__name__': 'ALERTS', 'alertname': 'ServiceCo...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
4,ALERTS,,,"{'__name__': 'ALERTS', 'alertname': 'ServiceCo...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
...,...,...,...,...,...,...,...
30742,scrape_series_added,clspromsg-edi,2,"{'Product': 'BIS', 'Service': 'CommunicationSe...","[[1602752343, 0], [1602752373, 0], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
30767,up,clspromsg-cfs,1,"{'Product': 'BIS', 'Service': 'CommunicationSe...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
30768,up,clspromsg-cfs,2,"{'Product': 'BIS', 'Service': 'CommunicationSe...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
30769,up,clspromsg-edi,1,"{'Product': 'BIS', 'Service': 'CommunicationSe...","[[1602752343, 1], [1602752373, 1], [1602752403...",1602752343,b18b1c2f-23fa-4e05-b192-184d36968dcb
